# Memory Consolidation
> Merge similar memories to reduce redundancy and keep storage efficient.

`SimilarityConsolidator` compares a new memory entry against existing ones
using embeddings. When similarity exceeds a threshold the entries are merged.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.memory.consolidation import SimilarityConsolidator
from anchor.models import MemoryEntry

## Mock Embedding Function
In production, use a real embedding model. Here we create a deterministic
hash-based mock.

In [ ]:
def mock_embed(text: str) -> list[float]:
    """Deterministic pseudo-embedding for demonstration."""
    padded = text[:128].ljust(128)
    return [hash(c) % 100 / 100.0 for c in padded]

# Quick test
vec = mock_embed("hello world")
print(f"Embedding dimension: {len(vec)}")
print(f"First 5 values: {vec[:5]}")

## Create the Consolidator

In [ ]:
consolidator = SimilarityConsolidator(
    embed_fn=mock_embed,
    similarity_threshold=0.85,
    max_cache_size=1000,
)

print(f"Similarity threshold: {consolidator.similarity_threshold}")
print(f"Max cache size: {consolidator.max_cache_size}")

## Prepare Memory Entries
Create a set of existing entries and a new entry to consolidate against them.

In [ ]:
from datetime import datetime, timezone

existing_entries = [
    MemoryEntry(
        entry_id="e-001",
        content="Python was released in 1991 by Guido van Rossum.",
        created_at=datetime(2024, 1, 1, tzinfo=timezone.utc),
    ),
    MemoryEntry(
        entry_id="e-002",
        content="JavaScript is the most popular web programming language.",
        created_at=datetime(2024, 1, 2, tzinfo=timezone.utc),
    ),
    MemoryEntry(
        entry_id="e-003",
        content="Rust provides memory safety without garbage collection.",
        created_at=datetime(2024, 1, 3, tzinfo=timezone.utc),
    ),
]

# New entry similar to e-001
new_entry = MemoryEntry(
    entry_id="e-new",
    content="Python was first released in 1991 by Guido van Rossum.",
    created_at=datetime(2024, 6, 1, tzinfo=timezone.utc),
)

print(f"Existing entries: {len(existing_entries)}")
print(f"New entry: '{new_entry.content[:50]}...'")

## Run Consolidation
The consolidator returns a list of `(MemoryOperation, MemoryEntry | None)` tuples.

In [ ]:
operations = consolidator.consolidate(new_entry, existing_entries)

print(f"Operations returned: {len(operations)}\n")
for op, entry in operations:
    entry_info = entry.entry_id if entry else "N/A"
    content_preview = entry.content[:60] if entry else ""
    print(f"  Operation: {op}")
    print(f"  Entry:     {entry_info}")
    if content_preview:
        print(f"  Content:   {content_preview}...")
    print()

## Key Takeaways
- `SimilarityConsolidator` deduplicates memories based on embedding similarity
- Supply your own `embed_fn` to control how content is vectorized
- `similarity_threshold` (0-1) controls how aggressive merging is
- Consolidation returns operations you can apply to your storage backend
- This prevents unbounded growth of near-duplicate memories

**Next:** [Garbage Collection](08_garbage_collection.ipynb)